# Japon : les actions du Nikkei 225 font-elles mieux que l'indice ?

Groupe 1 = bourse de Tokyo. On reprend le test du plafond (`ceiling_check.ipynb`) mais avec ~240 actions japonaises au lieu du seul Nikkei.

1. **Plafond** : dates connues, rendements réels du jour de la target → quelle accuracy ?
2. **Expositions qui bougent** : même chose en ré-estimant les expositions sur la dernière année (borne haute)
3. **Datation** : est-ce que les actions permettent de retrouver la date d'une ligne isolée (le vrai problème du test) ?

Il faut d'abord lancer `market_data/download_nikkei225.py` (crée `market_data/japan_returns.csv`).

⚠️ Ce n'est pas une prévision : on utilise les vrais rendements du jour de la target.

In [1]:
import numpy as np
import pandas as pd
import exchange_calendars as xcals
import warnings
warnings.filterwarnings("ignore")

X = pd.read_csv("data/X_train.csv", index_col="ROW_ID")
y = pd.read_csv("data/y_train.csv", index_col="ROW_ID")["target"]
dates = pd.read_csv("calendar/train_dates.csv", parse_dates=["date"])
stocks = pd.read_csv("market_data/japan_returns.csv", index_col="Date", parse_dates=True)
index_returns = pd.read_csv("market_data/returns.csv", index_col="Date", parse_dates=True)

# group 1 only, dates found with the holidays (reliable)
dates = dates[(dates["method"] == "holidays") & dates["date"].notna()]
X["date"] = X["TS"].map(dict(zip(dates["TS"], dates["date"])))
X["target"] = y
X1 = X[(X["GROUP"] == 1) & X["date"].notna()].copy()

sessions = pd.DatetimeIndex(xcals.get_calendar("XTKS", start="2004-01-01").sessions)
X1["next_date"] = sessions[sessions.searchsorted(X1["date"], side="right")]

print(len(X1), "rows of group 1,", X1["ALLOCATION"].nunique(), "allocations,",
      X1["date"].min().date(), "->", X1["date"].max().date())

99890 rows of group 1, 70 allocations, 2007-09-26 -> 2015-10-30


## Les actions sur la période

In [2]:
period = (stocks.index >= X1["date"].min()) & (stocks.index <= X1["next_date"].max())
coverage = stocks.loc[period].notna().mean()
print("stocks downloaded:", stocks.shape[1])
print("stocks with >= 90% of the days on the period:", int((coverage >= 0.9).sum()))

keep = coverage[coverage >= 0.9].index
# a few bad prints on Yahoo (moves > 30% in a day), we cap them
R = stocks[keep].clip(-0.3, 0.3).reindex(sessions).fillna(0)
R["NIKKEI225"] = index_returns["NIKKEI225"].reindex(sessions).fillna(0)

# same scale for every column, so one ridge penalty fits all
fit_days = sessions[(sessions >= X1["date"].min()) & (sessions < "2013-01-01")]
R = R / R.loc[fit_days].std().replace(0, 1)
stock_cols = list(keep)
print("columns used:", len(stock_cols), "stocks + the index")

stocks downloaded: 238
stocks with >= 90% of the days on the period: 219


columns used: 219 stocks + the index


## 1. Plafond : expositions fixes apprises sur 2007-2012, mesurées sur 2013-2015

Une régression ridge par allocation : `target(d) ~ rendements des actions à la séance suivante`. La pénalité est choisie sur 2012.

In [3]:
def ridge(A, b, lam):
    return np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ b)


def fit_and_predict(columns, lam, fit_end, pred_start, pred_end):
    # one ridge per allocation, fitted before fit_end, predictions on [pred_start, pred_end)
    pred = pd.Series(np.nan, index=X1.index)
    betas = {}
    for alloc, sub in X1.groupby("ALLOCATION"):
        rows_fit = sub[sub["date"] < fit_end]
        rows_pred = sub[(sub["date"] >= pred_start) & (sub["date"] < pred_end)]
        if len(rows_fit) < 100:
            continue
        A = R.loc[rows_fit["next_date"], columns].values
        b = rows_fit["target"].values
        beta = ridge(A, b, lam)
        betas[alloc] = beta
        if len(rows_pred) > 0:
            pred[rows_pred.index] = R.loc[rows_pred["next_date"], columns].values @ beta
    return pred, betas


def accuracy(pred, mask):
    ok = mask & pred.notna()
    return ((pred[ok] > 0) == (X1.loc[ok, "target"] > 0)).mean()


def r2(pred, mask):
    ok = mask & pred.notna()
    t = X1.loc[ok, "target"]
    return 1 - ((t - pred[ok]) ** 2).sum() / ((t - t.mean()) ** 2).sum()


# choose the penalty: fit before 2012, check on 2012
val = (X1["date"] >= "2012-01-01") & (X1["date"] < "2013-01-01")
val_r2 = {}
for lam in [10, 100, 1000, 10000, 100000]:
    p, _ = fit_and_predict(stock_cols, lam, "2012-01-01", "2012-01-01", "2013-01-01")
    val_r2[lam] = r2(p, val)
    print(f"lambda {lam:>6}: R2 on 2012 = {val_r2[lam]:.3f} | accuracy {accuracy(p, val):.1%}")

best_lam = max(val_r2, key=val_r2.get)
print("chosen lambda:", best_lam)

lambda     10: R2 on 2012 = -0.214 | accuracy 58.2%


lambda    100: R2 on 2012 = 0.009 | accuracy 59.6%


lambda   1000: R2 on 2012 = 0.055 | accuracy 59.5%


lambda  10000: R2 on 2012 = 0.031 | accuracy 56.7%


lambda 100000: R2 on 2012 = 0.012 | accuracy 55.4%
chosen lambda: 1000


In [4]:
hold = X1["date"] >= "2013-01-01"
results = {}
results["always 1"] = (X1.loc[hold, "target"] > 0).mean()

p_index, _ = fit_and_predict(["NIKKEI225"], 10, "2013-01-01", "2013-01-01", "2100-01-01")
results["Nikkei index only"] = accuracy(p_index, hold)

p_stocks, betas = fit_and_predict(stock_cols, best_lam, "2013-01-01", "2013-01-01", "2100-01-01")
results["Nikkei 225 stocks (fixed exposures)"] = accuracy(p_stocks, hold)

print(f"R2 2013-2015 : index {r2(p_index, hold):.3f} | stocks {r2(p_stocks, hold):.3f}")
pd.Series(results).round(4).to_frame("accuracy 2013-2015")

R2 2013-2015 : index 0.084 | stocks 0.111


,accuracy 2013-2015
always 1,0.4933
Nikkei index only,0.5575
Nikkei 225 stocks (fixed exposures),0.5900


## 2. Expositions qui bougent : ré-estimées chaque mois sur l'année précédente

Les portefeuilles changent (turnover). Ici on refait la régression tous les 20 jours sur les 250 jours d'avant. C'est une borne haute : sur le vrai test (2016-2026) on n'a pas ces données récentes.

In [5]:
hold_days = sorted(X1.loc[hold, "date"].unique())
p_rolling = pd.Series(np.nan, index=X1.index)

for k in range(0, len(hold_days), 20):
    start = hold_days[k]
    end = hold_days[min(k + 20, len(hold_days) - 1)] if k + 20 < len(hold_days) else pd.Timestamp("2100-01-01")
    window_start = start - pd.Timedelta(days=365)
    for alloc, sub in X1.groupby("ALLOCATION"):
        rows_fit = sub[(sub["date"] >= window_start) & (sub["date"] < start)]
        rows_pred = sub[(sub["date"] >= start) & (sub["date"] < end)]
        if len(rows_fit) < 100 or len(rows_pred) == 0:
            continue
        beta = ridge(R.loc[rows_fit["next_date"], stock_cols].values, rows_fit["target"].values, best_lam)
        p_rolling[rows_pred.index] = R.loc[rows_pred["next_date"], stock_cols].values @ beta

results["Nikkei 225 stocks (exposures re-fitted monthly)"] = accuracy(p_rolling, hold)
print(f"R2 2013-2015 with rolling exposures: {r2(p_rolling, hold):.3f}")
pd.Series(results).round(4).to_frame("accuracy 2013-2015")

R2 2013-2015 with rolling exposures: 0.162


,accuracy 2013-2015
always 1,0.4933
Nikkei index only,0.5575
Nikkei 225 stocks (fixed exposures),0.5900
Nikkei 225 stocks (exposures re-fitted monthly),0.6009


In [6]:
# is the prediction more often right when it predicts a big move ?
strength = p_rolling[hold].abs()
for q in [0.0, 0.5, 0.8]:
    m = hold & (p_rolling.abs() >= strength.quantile(q))
    print(f"top {100 - int(q * 100)}% strongest predictions: {accuracy(p_rolling, m):.1%}")

top 100% strongest predictions: 60.1%
top 50% strongest predictions: 65.9%
top 20% strongest predictions: 71.3%


## 3. Datation d'une ligne isolée avec les actions

Pour chaque date `TS` du groupe 1 (2013-2015), on fait comme si elle était isolée : on cherche la séance de Tokyo où l'historique de 20 jours des allocations est le mieux expliqué par les actions. Avec l'indice seul, on trouvait le bon jour ~1 fois sur 3 (tous groupes confondus).

In [7]:
hist_cols = [f"RET_{i}" for i in range(20, 0, -1)]   # oldest -> newest


def predicted_returns(betas, columns):
    # predicted return of each allocation on every Tokyo session
    M = R[columns].values
    return {alloc: M @ b for alloc, b in betas.items()}


def date_scores(rows, pred_by_alloc):
    # score of every Tokyo session as the date of these rows
    total = np.zeros(len(sessions) - 19)
    for alloc, hist in zip(rows["ALLOCATION"], rows[hist_cols].values):
        if alloc not in pred_by_alloc:
            continue
        p = pred_by_alloc[alloc]
        h = np.nan_to_num(hist.astype(float))
        # for a window starting at o: 2 * sum(h * p) - sum(p^2)  (= drop of the squared error)
        cross = np.convolve(p, h[::-1], mode="valid")
        energy = np.convolve(p ** 2, np.ones(20), mode="valid")
        total += 2 * cross - energy
    return pd.Series(total, index=sessions[19:])


# exposures learned before 2013 (fixed), for the stocks and for the index alone
_, betas_index = fit_and_predict(["NIKKEI225"], 10, "2013-01-01", "2013-01-01", "2013-01-01")
pred_index = predicted_returns(betas_index, ["NIKKEI225"])
pred_stocks = predicted_returns(betas, stock_cols)

test_like = X1.loc[hold].drop_duplicates("TS").sample(150, random_state=0)
found = {"index": [], "stocks": []}
for ts, true_date in zip(test_like["TS"], test_like["date"]):
    rows = X1[X1["TS"] == ts]
    for name, pred in [("index", pred_index), ("stocks", pred_stocks)]:
        s = date_scores(rows, pred)
        found[name].append(s.idxmax() == true_date)

for name in found:
    print(f"{name}: exact date found for {np.mean(found[name]):.0%} of 150 isolated dates (group 1 only)")

index: exact date found for 2% of 150 isolated dates (group 1 only)
stocks: exact date found for 8% of 150 isolated dates (group 1 only)
